# B3 — data-driven operator: new training set + DD/Sobolev training

**Follow-up paper, Professor Rabczuk's 2026-10-20 email**: "a fair PI-vs-DD comparison for B3... figure out when the expensive high-fidelity labels actually make PI worthwhile," and separately, "Sobolev/H1 training for the pure DD model" as one of the four candidate error-reduction methods. B3 has never had a trained data-driven baseline at all -- only the physics-informed DEM checkpoint (`train_B3.py`) exists, so a labeled training set has to be generated first.

**This cell does two things in sequence**:

1. **Generates a new, 200-sample FEM-labeled training set** for B3, using the project's own already-trusted `data_generate_B3_dataset.generate_dataset` (unchanged, no new FEM code) at `seed=50` -- clearly disjoint from the held-out evaluation set (`seed=99999`) and resumable (re-running this cell after an interruption picks up where it left off, same mechanism the held-out set generation already uses). **Watch the printed per-sample time on the first few samples** -- B3's 3D solve has never been timed for this purpose before, so this run's own log is the first real measurement; if it looks far more expensive than B1/B2's 2D ~25-60s/sample, interrupt and report the observed rate rather than letting 200 samples run unobserved.
2. **Trains two data-driven operators** on that new set -- `--loss rel_l2` (plain DD, needed for the PI-vs-DD comparison itself) and `--loss sobolev` (the gradient-matching variant Professor Rabczuk separately asked for) -- both evaluated against the exact same held-out set (`b3_dataset_clean_holdout`) every physics-informed B3 checkpoint has been evaluated on, at the same 50,000-step budget `train_B3.py`'s own baseline used, so every resulting number is directly comparable to the existing physics-informed results.

Both `train_B3_data_driven.py` (the training script) and this generation step were verified with a CPU smoke test (tiny synthetic data, all 4 loss variants) before any real GPU/FEM time was spent -- see PROJECT_STATUS.md.

In [ ]:
# =====================================================================
#  CELL -- B3 data-driven: (1) generate a new 200-sample labeled
#  training set (seed=50, disjoint from the held-out set), (2) train
#  plain DD and Sobolev operators, both evaluated against the existing
#  held-out set. See the markdown cell above for the full rationale.
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys
import time

_started = time.time()


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU)')
device_str = 'cuda' if torch.cuda.is_available() else None

R = '/content/drive/MyDrive/pfem_run'
DD_DATASET_DIR = f'{R}/b3_dd_training_set'
HOLDOUT_DATASET = f'{R}/b3_dataset_clean_holdout/dataset.h5'
os.makedirs(DD_DATASET_DIR, exist_ok=True)

print(f'\n{"=" * 90}')
print('=== Step 1: generating 200-sample labeled training set (seed=50) ===')
from omar_pfem.data.data_generate_B3_dataset import generate_dataset

t0 = time.time()
generate_dataset(num_samples=200, output_dir=DD_DATASET_DIR, seed=50, device=device_str)
gen_elapsed = time.time() - t0
print(f'Dataset generation finished in {gen_elapsed:.1f}s '
      f'({gen_elapsed / 200:.2f}s/sample average).')

TRAIN_DATASET = f'{DD_DATASET_DIR}/dataset.h5'

for loss in ['rel_l2', 'sobolev']:
    print(f'\n{"=" * 90}')
    print(f'=== Step 2: training data-driven B3 operator, --loss {loss} ===')
    OUT_DIR = f'{R}/b3_data_driven_{loss}'
    sys.argv = [
        'train_B3_data_driven.py',
        '--train_dataset', TRAIN_DATASET,
        '--test_dataset', HOLDOUT_DATASET,
        '--out_dir', OUT_DIR,
        '--loss', loss,
        '--opt_steps', '50000',
        '--batch_size', '8',
        '--eval_every', '2000',
    ]
    from omar_pfem.train_B3_data_driven import main as train_dd_main
    train_dd_main()

    try:
        from omar_pfem.run_manifest import write_manifest
        write_manifest(OUT_DIR, kind=f'b3_data_driven_{loss}',
                        args={'loss': loss, 'opt_steps': 50000, 'batch_size': 8,
                              'train_dataset': TRAIN_DATASET, 'test_dataset': HOLDOUT_DATASET},
                        started_at=_started,
                        outputs=[f'{OUT_DIR}/model_best.pt', f'{OUT_DIR}/data_driven_B3_{loss}.json'],
                        notes="First data-driven B3 training run (Professor Rabczuk's "
                              "2026-10-20 email: fair PI-vs-DD comparison for B3, and "
                              "Sobolev/H1 training for the DD model). Trained on a new "
                              "200-sample set (seed=50), evaluated on the same held-out "
                              "set every physics-informed B3 checkpoint uses, for a "
                              "directly comparable number.")
    except Exception as e:
        print(f'[manifest] not recorded: {e}')

print('\nDone. Send this cell\'s full printed output back -- especially both runs\' '
      'final "best validation rel. L2" lines, to compare against the physics-informed '
      'checkpoint\'s own combined displacement error.')
